In [1]:
import pandas as pd
from pathlib import Path

from modules.access_control import final_access_decision

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

recommendations = pd.read_csv(RESULTS_DIR / "top_n_recommendations_user_0.csv")
flagged_users_df = pd.read_csv(RESULTS_DIR / "flagged_users.csv")

flagged_users = set(flagged_users_df["user_id"].tolist())

print("Recommendations shape:", recommendations.shape)
print("Flagged users:", len(flagged_users))

recommendations

Recommendations shape: (5, 7)
Flagged users: 34


,user_id,service_id,direct_trust,indirect_trust,final_trust,recommendation_rank,status
0,0,19,0.985082,0.985362,0.985194,1,Recommended
1,0,3,0.984934,0.983564,0.984386,2,Recommended
2,0,4,0.984567,0.983946,0.984319,3,Recommended
3,0,1,0.984484,0.983262,0.983995,4,Recommended
4,0,2,0.983538,0.981418,0.982690,5,Recommended


In [2]:
rbac_policy = {
    "admin": ["ALL"],
    "registered_user": [0, 1, 2, 3, 4, 5, 6, 7],
    "guest": [0, 1]
}

user_roles = {
    0: "registered_user"
}

target_user = 0
user_role = user_roles[target_user]

print("Target user:", target_user)
print("User role:", user_role)
print("RBAC policy:", rbac_policy)

Target user: 0
User role: registered_user
RBAC policy: {'admin': ['ALL'], 'registered_user': [0, 1, 2, 3, 4, 5, 6, 7], 'guest': [0, 1]}


In [3]:
access_results = []

for _, row in recommendations.iterrows():
    service_id = int(row["service_id"])
    final_trust_score = float(row["final_trust"])

    decision = final_access_decision(
        user_id=target_user,
        user_role=user_role,
        service_id=service_id,
        final_trust_score=final_trust_score,
        rbac_policy=rbac_policy,
        trust_threshold=0.60,
        flagged_users=set()  # set() used here because recommendation test was for sample user 0
    )

    access_results.append(decision)

access_df = pd.DataFrame(access_results)

access_df

,user_id,service_id,user_role,final_trust_score,access_decision,reason
0,0,19,registered_user,0.985194,Access Denied,Role-policy mismatch
1,0,3,registered_user,0.984386,Access Granted,Trusted recommendation and valid RBAC permission
2,0,4,registered_user,0.984319,Access Granted,Trusted recommendation and valid RBAC permission
3,0,1,registered_user,0.983995,Access Granted,Trusted recommendation and valid RBAC permission
4,0,2,registered_user,0.982690,Access Granted,Trusted recommendation and valid RBAC permission


In [4]:
access_df.to_csv(
    RESULTS_DIR / "access_decisions_user_0.csv",
    index=False
)

print("Saved:", RESULTS_DIR / "access_decisions_user_0.csv")

Saved: results/access_decisions_user_0.csv


In [5]:
access_summary = access_df["access_decision"].value_counts().reset_index()
access_summary.columns = ["access_decision", "count"]

access_summary.to_csv(
    RESULTS_DIR / "access_decision_summary_user_0.csv",
    index=False
)

access_summary

,access_decision,count
0,Access Granted,4
1,Access Denied,1


In [6]:
sample_flagged_user = list(flagged_users)[0]

blocked_test = final_access_decision(
    user_id=sample_flagged_user,
    user_role="admin",
    service_id=0,
    final_trust_score=0.99,
    rbac_policy=rbac_policy,
    trust_threshold=0.60,
    flagged_users=flagged_users
)

blocked_test

{'user_id': 261,
 'service_id': 0,
 'user_role': 'admin',
 'final_trust_score': 0.99,
 'access_decision': 'Access Denied',
 'reason': 'User is flagged as malicious or low-trust'}